[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CityScope/TransitLOSStudies/blob/main/example_metro_area/example.ipynb)

# Transit level of service for any area

Pick an area anywhere in the world (a city, a metropolitan area, or a shape you draw), and get its streets, H3 hexagons and census data with a **transit level of service** for every place, an interactive map, and a zip to download.

## How to use this notebook

**Quick start:** change the place in *2. Settings*, then **Runtime → Run all** (Jupyter: Run → Run All Cells). Run all takes the defaults everywhere and never stops to ask. The first run installs the packages, a few minutes more. After that a municipality takes a few minutes and a whole metro area longer.

**To choose more**, run the cells one at a time instead. Every step below explains its choices; the optional ones are:

| Step | What you can choose |
|---|---|
| 3. The area | a place name and area type, a shape you draw on a map, or your own file |
| 4. Transit feeds | which feeds to download, chosen from their metadata, and which of those to use |
| 5. Service date | the day analysed, with a plot of the service on every date |
| 2. Settings | day type and hours |
| 6. Census sources | which census sources to download (WorldPop included): the unticked ones are never downloaded |
| 9. Census columns | which census columns to keep in the download |

**About the hidden code:** in Colab most cells are *forms* with their code collapsed, so you see only the fields and the results. Everything you need to know is written in these text cells. To see a cell's code, double-click its title (or View → Show/hide code); you do not need to.

**Where the data comes from:** the most local source available: the area's own census and official transit feeds when there are, otherwise global data (WorldPop, Mobility Database, Transitland). The level of service uses the standard scoring functions of the transitLOS package, taken from the transit literature; nothing is fitted to data.

## 1. Set up

Run this cell first (Run all does). It gets the study code and, where the packages are missing (always in a new Colab session), installs them the first time into a private environment, so **the notebook never restarts**. You will see "Installing the study packages" for a few minutes on the first run only.

In [ ]:
#@title 1. Set up (installs on the first run) { display-mode: "form" }
import subprocess, sys
from pathlib import Path
if not Path("bootstrap.py").is_file():                 # not inside the study folder (Colab): use the clone, making it the first time
    folder = Path("TransitLOSStudies/example_metro_area")
    if not folder.is_dir():
        subprocess.check_call(["git", "clone", "-q", "--depth", "1", "https://github.com/CityScope/TransitLOSStudies.git"])
    sys.path.insert(0, str(folder.resolve()))
import bootstrap
study = bootstrap.setup()
import ui

## 2. Settings

The only cell you need to edit. In Colab it is a form with these fields: change the values, then run the cell. (Your changes are read when the later cells run, so after changing a value run again from the step it affects.)

- **`PLACE`**: the place to study, as you would search for it: a name with its region and country, e.g. `Cambridge, Massachusetts, USA`, `Guadalajara, Mexico`, `Hamburg, Germany`. Ignored when you give `AOI_FILE` or tick `DRAW`.
- **`AREA`**: how much around the place to take.
  - `municipality`: the city's own boundary (the official one when the country publishes it).
  - `urban_core`: the UN urban centre, the dense built-up city.
  - `metro_area`: the UN functional urban area, the urban centre plus its commuting zone (largest, slowest).
- **`AOI_FILE`**: your own area instead of a place name. Empty (default), the path of a `.gpkg`, `.geojson`, `.kml` or zipped shapefile, or `upload` to pick a file from your computer (Colab).
- **`DRAW`**: draw the area yourself on a map. Unticked (default) or ticked; see step 3.
- **`DAY`**: the type of day analysed: `weekday` (a working day), `saturday` or `sunday`. Step 5 then suggests a date of this type, avoiding public holidays.
- **`START_TIME`, `END_TIME`**: the hours analysed, 24-hour clock `HH:MM`, default `06:00` to `22:00`. Service outside these hours does not count.
- **`FEEDS`**: where the transit feeds (GTFS) come from. `download` (default) finds the area's feeds online, the country's official portal first, then the Mobility Database and Transitland. Or give the folder with your own feeds, zips or unzipped (in Colab, upload them first with the Files panel on the left). The feeds are saved in the area's results folder (see step 6).
- **API keys**: nothing to set here. Some sources need a free key; when one is missing, the notebook shows a box with the sign-up link where you paste it (it is saved and not asked again):
  - US census (ACS and decennial): US Census Bureau key, https://api.census.gov/data/key_signup.html (arrives by email).
  - German census: Destatis GENESIS-Online token, https://www.destatis.de/EN/Service/OpenData/api-webservice.html.
  - Spain's National Access Point (feeds): https://nap.transportes.gob.es.
  - Transitland (feeds): https://www.transit.land.
  - Mobility Database (feeds, optional): https://mobilitydatabase.org/sign-up.

  In Colab you can also add a key as a **Secret** (key icon in the left bar) named `US_CENSUS_API_KEY`, `TRANSITLAND_API_KEY`, `NAP_API_KEY`, `DESTATIS_GENESIS_TOKEN`, or `MOBILITY_DATABASE_REFRESH_TOKEN`, with notebook access allowed.


In [ ]:
#@title Settings { display-mode: "form" }
# ----- Area: a place name and type, or your own file, or draw it in step 3
PLACE = "Cambridge, Massachusetts, USA"  #@param {type:"string"}
AREA = "municipality"  #@param ["metro_area", "urban_core", "municipality"]
AOI_FILE = ""  #@param {type:"string"}
DRAW = False  #@param {type:"boolean"}
# AREA: metro_area = UN functional urban area (urban centre + commuting zone), urban_core = UN urban centre, municipality = the city's boundary
# AOI_FILE: a .gpkg, .geojson, .kml or zipped shapefile (its path; "upload" picks one from your computer in Colab). DRAW: draw the area on a map

# ----- When
DAY = "weekday"  #@param ["weekday", "saturday", "sunday"]
START_TIME = "06:00"  #@param {type:"string"}
END_TIME = "22:00"  #@param {type:"string"}

# ----- Data
FEEDS = "download"  #@param {type:"string"}
# FEEDS: "download" finds the area's feeds online, or the folder with your GTFS feeds (a Python list of zips, folders and links also works)

## 3. The area

The map shows the area that will be analysed (red outline), with its name and size. Check it before the long steps. There are three ways to get an area:

- **A place name** (default): set `PLACE` and `AREA` in Settings and run the cells. Nothing else to do.
- **Your own file:** set `AOI_FILE` in Settings. With `upload` a button appears in Colab to choose the file from your computer.
- **Draw it:** tick `DRAW` in Settings, run the first cell below, and a map opens. Click the **polygon button** (top left of the map), click the corners of your area and double-click to close it (the bin button deletes it). Then run the second cell, which shows the area you drew.

If the map shows the wrong place, change the settings and run this step again. The outline is simplified for display only; the full shape is used in the study.

In [ ]:
#@title Draw the area (only with DRAW) { display-mode: "form" }
if DRAW:
    draw_map = ui.draw_area(study.area_from_place(PLACE, AREA))
    display(draw_map)

In [ ]:
#@title The area { display-mode: "form" }
if DRAW:
    assert draw_map.drawn, "Nothing drawn: draw a polygon on the map above, then run this cell and the ones below again."
    aoi = study.area_from_geojson(draw_map.drawn[-1])
else:
    aoi = study.get_area(PLACE, AREA, ui.upload_file() if AOI_FILE == "upload" else AOI_FILE or None)
ui.show_area(aoi)

## 4. Transit feeds

Feeds are chosen in three steps, so that you download only what you need.

**4a. Choose the databases.** The global ones (Mobility Database, Transitland) and those of the area's country are ticked; untick any you do not want. The ones that need a free key show a box to paste it, with the sign-up link.

**4b. Find the feeds (nothing is downloaded yet).** The sources (the country's official portal, the Mobility Database, Transitland) list the feeds for the area. Each line shows the agency, the sources that list it, where it is based, its transport modes and when it was last updated, and **why it is or is not thought to serve the area**: whether its coverage includes the area and is not regional or intercity in size, or whether the agency is based in the area. **Those that probably serve the area start ticked**; the others (intercity coaches, agencies of other cities) are listed unticked in case you need them. Tick and untick, then run the next cell. The same feed listed by several sources appears once.

**4c. Download the ticked feeds.** Only the ticked feeds are downloaded, then listed again with what was found in the area: how many **stops are inside it** and how many **trips run on a typical day**, and the period the calendar covers. Feeds with no stop in the area are dropped. **Untick any you do not want** to leave out; the next steps use the ticked ones.

Run all takes the defaults: the feeds that probably serve the area. Some sources need a free API key (see Settings); without one they are skipped and fewer feeds are listed. With your own feeds (`FEEDS` in Settings) step 4a is skipped.

Some feed sources need a free account key (Spain's National Access Point, Transitland, and the Mobility Database refresh token, which is optional). The cell below shows a box with the sign-up link for each one that is missing; paste the key and press *Save key* **before** running 4a, otherwise 4a leaves that source out.


In [ ]:
#@title 4a. Which feed databases to search { display-mode: "form" }
database_choice = ui.select_databases(study.feed_databases(aoi))   # the global ones and the country's own, all ticked
keys_box = ui.ask_keys(study, aoi, feeds=database_choice)   # asks for the API keys they need
display(database_choice, keys_box)

In [ ]:
#@title 4b. Find the feeds (no download) { display-mode: "form" }
candidates = study.find_feeds(aoi, FEEDS, sources=database_choice.value or None)
candidate_choice = ui.select_candidates(candidates)
candidate_choice

In [ ]:
#@title 4c. Download the chosen feeds { display-mode: "form" }
feeds = study.list_feeds(aoi, FEEDS, sources=database_choice.value or None, selected=candidate_choice.value)
feed_choice = ui.select_feeds(feeds)
feed_choice

## 5. Service date

The study analyses **one typical day**. The plot shows the service of the ticked feeds on every date, as stops × trips in the area (the height of a bar is the number of times a vehicle calls at a stop of the area that day).

**Colours:** blue weekdays, amber Saturdays, purple Sundays, red hatched public holidays (with their names). The day type you set in Settings is drawn strongest. A dashed line marks the **suggested date**, a red line the date that will be used.

**The suggestion** is a typical day of your day type within 60 days of today: the middle date among those with at least 95% of the busiest such day's service, holidays excluded. A schedule change shows as a step in the bars, so check that the suggested date is on the schedule you want.

**To use another date,** pick it in the **Service date** box above the plot (the red line follows it) and run the next steps. A date you choose is used as it is, even on a holiday; a feed with no service on that date is scored on its own typical day. Run all uses the suggestion.

In [ ]:
#@title Suggested service date { display-mode: "form" }
dates = study.service_dates(aoi, feed_choice.value, DAY)
date_choice = ui.pick_date(dates)
date_choice

## 6. Census sources

The census comes from several **sources, and each is a separate download**. Tick the ones you want; **the unticked ones are never downloaded**, which saves time. Run all takes the defaults.

- **Nothing is locked.** The **census population** (the decennial census blocks in the USA, the national census elsewhere) is ticked by default where the census is sub-municipal. If you untick it, **WorldPop's population grid is ticked for you**, and where a country has no census WorldPop is the population source. The buildings (GlobalBuildingAtlas) are downloaded automatically whenever the run needs them (the built-surface population and resampling), so they are not in the list.
- **ACS 5-year** holds most of the social and economic columns (income, housing, education, commuting, jobs); **jobs by workplace**, **religion**, **housing in earlier periods** and **elections** each add their own columns.
- **WorldPop** has two groups. The *population grid* is only needed where the census areas are large or the census population is unticked (with a fine census, such as US block groups, the census gives the population, so it is not downloaded). The *age, sex and urban layers* are a one-time download of **tens of GB** for the USA and are unticked by default.
- Other countries show their national census as one group.

Which columns of a downloaded source to keep is chosen later, in step 9.

Some sources need a **free API key** (the US Census Bureau's, for ACS and the decennial census). If one is missing, a box appears below the list with the link to sign up for it: paste the key there and press *Save key*. It is saved on this machine (in Colab: this session) and not asked again. You can also add it as the Colab Secret `US_CENSUS_API_KEY`.


In [ ]:
#@title Census sources: what to download { display-mode: "form" }
sources = study.census_sources(aoi)
source_choice = ui.select_sources(sources)
key_box = ui.ask_keys(study, aoi, source_choice)   # asks for the API keys the ticked sources need
display(source_choice, key_box)

## 7. Run

This gets the streets (OpenStreetMap) and the census, computes the **transit level of service** at every street and H3 hexagon, and builds the map and the zip. It prints its progress below; leave it running. Expect a few minutes for a municipality, up to an hour or more for a metro area.

Population comes from the census when its areas are small enough (the median area at most 1 km²); otherwise from WorldPop, which is then downloaded. Everything is saved in the area's results folder `areas/<name>/` next to the notebook (in Colab: `/content/TransitLOSStudies/example_metro_area/areas/<name>/`): the feeds in `gtfs_all`, the streets, the map `map.html` and the results zip. Copy that folder to Google Drive to keep it, because a Colab session is deleted after a while.

In [ ]:
#@title Run the study { display-mode: "form" }
zip_path = study.run(aoi, day=DAY, start_time=START_TIME, end_time=END_TIME, feed_names=feed_choice.value, service_date=date_choice.value, census_sources=source_choice.value)
print(zip_path)

## 8. The results on the map

The map shows the finest H3 hexagons (or circles) and the finest census level coloured by transit level of service, with the stops and routes. Use the menus on the map to change what is shown; with the circles shape, the **Circle size** slider scales the circles. The statistics panel has its own cell below, with more room. If the map appears blank, wait a few seconds: the map tiles load as it opens.

In [ ]:
#@title Interactive map { display-mode: "form" }
ui.show_map(zip_path)

## 9. Statistics

The statistics panel of the map, in a cell of its own so it has the room it needs: the **distribution** of the level of service (by population, area or any column), the **regression** against the census columns, the **ANOVA** census comparison, the **discretization** and the **metadata** (what each column is). Pick the area, scenario and column in the panel.

In [ ]:
#@title Statistics panel { display-mode: "form" }
ui.show_stats(zip_path)

## 10. Census columns and download

The results hold the census columns of the sources you downloaded, in **groups (one per source, plus the indicators this study derives from them)**, all ticked. Columns of a source you left out in step 6 are not there. Run the next cell to see the groups, then:

- **Untick a group** to leave all its columns out of the download.
- **Press *Show columns*** to open a group and see each column: its name, its description, its unit, the **census level** it was published at (block, block group, county, 100 m grid, or "derived" for columns made by this study), its source, and the result tables that hold it. Tick or untick single columns.
- The **search box** filters the columns by name or description; *Tick all* and *Untick all* act on the columns shown.

Then run the last cell, which writes the zip with only the ticked columns and starts the download (in Colab, the browser asks to save it). The streets, ids, population and level of service are always kept, and Run all keeps every column.

The zip holds `streets.parquet` (every street segment with its level of service), `h3_res<N>.parquet` (H3 hexagons at each resolution, with population, level of service and the census columns) and `census_<level>.parquet` (the census at its finest level). Read them with `pandas.read_parquet` (or `geopandas.read_parquet` for the tables with geometry).

In [ ]:
#@title Census column groups { display-mode: "form" }
columns = study.result_columns(aoi)
column_choice = ui.select_columns(columns)
column_choice

In [ ]:
#@title Download the results { display-mode: "form" }
zip_path = study.export_results(aoi, column_choice.value)
ui.download(zip_path)